### Base de Datos (Database)

#### Objetivos:

1. Documentacion sobre Spark SQL
2. Crear la Base de Datos "demo"
3. Acceder a "Calatalog" en la "Interfaz de Usuario"
4. Comando "SHOW"
5. Comando "DESCRIBE(DESC)"
6. Mostrar la Base de Datos Actual

In [0]:
CREATE SCHEMA IF NOT EXISTS demo;

In [0]:
SHOW DATABASES;

In [0]:
DESCRIBE DATABASE demo;

In [0]:
DESCRIBE DATABASE EXTENDED demo;

In [0]:
-- mostrar db actual
SELECT current_database();

In [0]:
SHOW TABLES IN demo;

In [0]:
-- cambio de DB
USE demo; 

In [0]:
-- consulto tablas de otra db
SHOW TABLES IN default;

### Tablas Administradas (Managed Tables)

#### Objetivos:

1. Crear una "Tabla Administrada (Managed Table)" con Python
2. Crear una "Tabla Administrada (Managed Table)" con SQL
3. Efecto de eliminar una Tabla Administrada
4. Describir(DESCRIBE) la Tabla

In [0]:
%run "../includes/configuration"

In [0]:
%python
results_movie_genre_language = spark.read.parquet(f"{gold_folder_path}/results_movie_genre_language")

In [0]:
%python
display(results_movie_genre_language)

In [0]:
%python
results_movie_genre_language.write.format("delta").saveAsTable("demo.results_movie_genre_language_python")

In [0]:
USE demo;
SHOW TABLES;

In [0]:
DESCRIBE EXTENDED results_movie_genre_language_python;

In [0]:
-- podemos hacer una consulta
SELECT * 
FROM results_movie_genre_language_python
WHERE genre_name = "Adventure";

In [0]:
-- podemos crear una tabla administrada a partir de una consulta
CREATE TABLE demo.results_movie_genre_language_sql
AS
SELECT * 
FROM results_movie_genre_language_python
WHERE genre_name = "Adventure";

In [0]:
Select * from results_movie_genre_language_sql;

In [0]:
SELECT current_database();

In [0]:
DESCRIBE EXTENDED results_movie_genre_language_sql;

In [0]:
-- elimino la tabla administrada .. SE ELIMINAN DATOS Y METADATOS
DROP TABLE IF EXISTS results_movie_genre_language_sql;

In [0]:
SHOW TABLES IN demo;

### Tablas External (External Tables)

#### Objetivos:

1. Crear una "Tabla Externa (External Table)" con Python
2. Crear una "Tabla Externa (External Table)" con SQL
3. Efecto de eliminar una Externa (External Table)
4. Describir(DESCRIBE) la Tabla

In [0]:
%python
results_movie_genre_language.write.format("delta").option("path", f"{gold_folder_path}/results_movie_genre_language_py").saveAsTable("demo.results_movie_genre_language_py")

In [0]:
-- ver en detalles que ahora es una tabla Externa
DESCRIBE EXTENDED results_movie_genre_language_py;

In [0]:
Select * from demo.results_movie_genre_language_py

In [0]:
-- podemos crear una tabla externa a partir de una consulta
CREATE TABLE demo.results_movie_genre_language_sql(
    title STRING,
    durantion_time INT,
    release_date DATE,
    vote_average FLOAT,
    language_name STRING,
    genre_name STRING,
    created_date TIMESTAMP
)
USING PARQUET
LOCATION "abfss://gold@moviehistory22.dfs.core.windows.net/results_movie_genre_language_ext_sql/"

In [0]:
SHOW TABLES In demo;

In [0]:
-- inserto la tabla
Insert into demo.results_movie_genre_language_sql
Select * from demo.results_movie_genre_language_py
Where genre_name = "Adventure";

In [0]:
select count(1)
from demo.results_movie_genre_language_sql

In [0]:
-- elimino la tabla creada
DROP TABLE demo.results_movie_genre_language_sql;

In [0]:
show tables in demo;

### Vistas (Views)

#### Objetivos

1. Crear Vista temporal
2. Crear Vista temporal global
3. Crear Vista Permanente

In [0]:
Select current_database()

In [0]:
-- vista temporal / no ocupa espacio en el disco... sirve para una representacion de una consulta

CREATE OR REPLACE TEMP VIEW v_results_movie_genres_language 
AS
select * from demo.results_movie_genre_language_py
Where genre_name = "Adventure";

In [0]:
Select * from v_results_movie_genres_language

-- se accede solo desde este notebook

In [0]:
-- ahora como es una vista temporal global

CREATE OR REPLACE GLOBAL TEMP VIEW gv_results_movie_genres_language 
AS
select * from demo.results_movie_genre_language_py
Where genre_name = "Drama";

In [0]:
SHOW tables in global_temp

In [0]:
select * from global_temp.gv_results_movie_genres_language

-- se puede consultar desde otroa notebooks

In [0]:
-- crear una vista permanente  ... quitar el temp

CREATE OR REPLACE VIEW pv_results_movie_genres_language 
AS
select * from demo.results_movie_genre_language_py
Where genre_name = "Comedy";

In [0]:
SHOW tables;

-- ahora esta dentro de demo... por eso se considera permanente

In [0]:
Select * from pv_results_movie_genres_language